# Domain LoRA - Overall EER Computation

각 도메인별 LoRA score 파일(D1~D7)을 합쳐서 전체 EER을 계산합니다.
- D1: `out/hp_search/{model}_lr{lr}_r{r}_a{alpha}/`
- D2~D7: `out/domain_lora/{model}/D{d}_lr{lr}_r{r}_a{alpha}/`

In [1]:
import pandas as pd
import os
import sys
sys.path.insert(0, '.')
from evaluate_metrics import compute_eer

DOMAIN_TO_AUGTYPES = {
    0: ['clean'],
    1: ['background_noise', 'background_music'],
    2: ['auto_tune'],
    3: ['high_pass_filter', 'low_pass_filter'],
    4: ['echo'],
    5: ['pitch_shift', 'time_stretch'],
    6: ['gaussian_noise'],
    7: ['reverberation'],
}

# NC protocols
PROTO_ASV19 = 'protocols/asv19_eval.txt'
PROTO_DF21 = 'protocols/df21_eval.txt'

## AASIST - HP 조합별 전체 EER

In [2]:
# AASIST HP 조합 (D1 HP search에서 선정된 3개)
aasist_hps = [
    {'lr': '5e-05', 'r': 16, 'alpha': 32},
    {'lr': '0.0001', 'r': 32, 'alpha': 32},
    {'lr': '5e-05', 'r': 32, 'alpha': 64},
]

def get_score_paths(model, hp, domains=range(1, 8)):
    """Get score file paths for all domains."""
    lr, r, alpha = hp['lr'], hp['r'], hp['alpha']
    paths = {}
    for d in domains:
        if d == 1:
            base = f'out/hp_search/{model}_lr{lr}_r{r}_a{alpha}'
        else:
            base = f'out/domain_lora/{model}/D{d}_lr{lr}_r{r}_a{alpha}'
        paths[d] = {
            'asv19': os.path.join(base, f'asv19_d{d}_scores.txt'),
            'df21': os.path.join(base, f'df21_d{d}_scores.txt'),
        }
    return paths

def merge_scores(score_paths, dataset='asv19'):
    """Merge score files from all domains into one DataFrame."""
    dfs = []
    for d, paths in score_paths.items():
        path = paths[dataset]
        if os.path.exists(path):
            df = pd.read_csv(path, sep=' ', header=None, names=['utt', 'spoof', 'bonafide'])
            df['domain'] = d
            dfs.append(df)
        else:
            print(f'  [WARN] Missing: {path}')
    if dfs:
        return pd.concat(dfs, ignore_index=True)
    return pd.DataFrame()

def compute_overall_eer(merged_scores, proto_path):
    """Compute overall EER from merged scores and protocol."""
    proto = pd.read_csv(proto_path, sep=' ', header=None, names=['filepath', 'label', 'aug_type'])
    merged = pd.merge(proto, merged_scores, left_on='filepath', right_on='utt')
    bf = merged[merged['label'] == 'bonafide']['bonafide'].values
    sp = merged[merged['label'] == 'spoof']['bonafide'].values
    if len(bf) == 0 or len(sp) == 0:
        return None, 0, 0
    eer, thr = compute_eer(bf, sp)
    return eer * 100, len(bf), len(sp)

def compute_domain_eers(merged_scores, proto_path):
    """Compute per-domain EER from merged scores."""
    proto = pd.read_csv(proto_path, sep=' ', header=None, names=['filepath', 'label', 'aug_type'])
    merged = pd.merge(proto, merged_scores, left_on='filepath', right_on='utt')
    results = {}
    for d_id in range(1, 8):
        allowed = DOMAIN_TO_AUGTYPES[d_id]
        dm = merged[merged['aug_type'].isin(allowed)]
        bf = dm[dm['label'] == 'bonafide']['bonafide'].values
        sp = dm[dm['label'] == 'spoof']['bonafide'].values
        if len(bf) > 0 and len(sp) > 0:
            eer, _ = compute_eer(bf, sp)
            results[f'D{d_id}'] = eer * 100
        else:
            results[f'D{d_id}'] = None
    return results

In [3]:
# AASIST 결과
print('=' * 80)
print('AASIST - Overall EER (D1~D7 LoRA scores merged)')
print('=' * 80)

aasist_results = []

for hp in aasist_hps:
    lr, r, alpha = hp['lr'], hp['r'], hp['alpha']
    print(f'\n--- lr={lr}, r={r}, alpha={alpha} ---')
    
    score_paths = get_score_paths('aasist', hp)
    
    for dataset, proto in [('asv19', PROTO_ASV19), ('df21', PROTO_DF21)]:
        merged = merge_scores(score_paths, dataset)
        if merged.empty:
            print(f'  {dataset.upper()}: No scores')
            continue
        
        overall_eer, n_bf, n_sp = compute_overall_eer(merged, proto)
        domain_eers = compute_domain_eers(merged, proto)
        
        print(f'  {dataset.upper()} Overall EER: {overall_eer:.4f}% (bf={n_bf}, sp={n_sp})')
        for dk, dv in domain_eers.items():
            print(f'    {dk}: {dv:.4f}%' if dv is not None else f'    {dk}: -')
        
        aasist_results.append({
            'model': 'aasist', 'lr': lr, 'r': r, 'alpha': alpha,
            'dataset': dataset, 'overall_eer': overall_eer,
            **domain_eers
        })

AASIST - Overall EER (D1~D7 LoRA scores merged)

--- lr=5e-05, r=16, alpha=32 ---
  ASV19 Overall EER: 5.6919% (bf=6623, sp=57489)
    D1: 3.9875%
    D2: 0.5202%
    D3: 2.7375%
    D4: 1.7666%
    D5: 8.9687%
    D6: 3.6069%
    D7: 10.2160%
  DF21 Overall EER: 13.0875% (bf=13357, sp=467178)
    D1: 6.0520%
    D2: 8.0573%
    D3: 15.2810%
    D4: 8.0667%
    D5: 14.8465%
    D6: 13.4957%
    D7: 16.8340%

--- lr=0.0001, r=32, alpha=32 ---
  ASV19 Overall EER: 4.8319% (bf=6623, sp=57489)
    D1: 3.4890%
    D2: 0.6483%
    D3: 1.7205%
    D4: 1.3752%
    D5: 7.6044%
    D6: 3.3589%
    D7: 11.5385%
  DF21 Overall EER: 12.1192% (bf=13357, sp=467178)
    D1: 6.4400%
    D2: 5.2631%
    D3: 11.2612%
    D4: 8.0763%
    D5: 14.7782%
    D6: 9.9238%
    D7: 16.4987%

--- lr=5e-05, r=32, alpha=64 ---
  ASV19 Overall EER: 4.8618% (bf=6623, sp=57489)
    D1: 4.4152%
    D2: 0.7842%
    D3: 1.5767%
    D4: 1.7666%
    D5: 8.2350%
    D6: 3.9945%
    D7: 9.6839%
  DF21 Overall EER: 12.4891% (b

## ConformerTCM - HP 조합별 전체 EER

In [4]:
# ConformerTCM HP 조합
conformer_hps = [
    {'lr': '0.0001', 'r': 16, 'alpha': 32},
    {'lr': '0.0001', 'r': 16, 'alpha': 16},
    {'lr': '0.0001', 'r': 4, 'alpha': 8},
]

print('=' * 80)
print('ConformerTCM - Overall EER (D1~D7 LoRA scores merged)')
print('=' * 80)

conformer_results = []

for hp in conformer_hps:
    lr, r, alpha = hp['lr'], hp['r'], hp['alpha']
    print(f'\n--- lr={lr}, r={r}, alpha={alpha} ---')
    
    score_paths = get_score_paths('conformertcm', hp)
    
    for dataset, proto in [('asv19', PROTO_ASV19), ('df21', PROTO_DF21)]:
        merged = merge_scores(score_paths, dataset)
        if merged.empty:
            print(f'  {dataset.upper()}: No scores')
            continue
        
        overall_eer, n_bf, n_sp = compute_overall_eer(merged, proto)
        domain_eers = compute_domain_eers(merged, proto)
        
        print(f'  {dataset.upper()} Overall EER: {overall_eer:.4f}% (bf={n_bf}, sp={n_sp})')
        for dk, dv in domain_eers.items():
            print(f'    {dk}: {dv:.4f}%' if dv is not None else f'    {dk}: -')
        
        conformer_results.append({
            'model': 'conformertcm', 'lr': lr, 'r': r, 'alpha': alpha,
            'dataset': dataset, 'overall_eer': overall_eer,
            **domain_eers
        })

ConformerTCM - Overall EER (D1~D7 LoRA scores merged)

--- lr=0.0001, r=16, alpha=32 ---
  ASV19 Overall EER: 5.7372% (bf=6623, sp=57489)
    D1: 4.8352%
    D2: 1.4325%
    D3: 2.7375%
    D4: 2.1736%
    D5: 9.9178%
    D6: 3.6147%
    D7: 10.3372%
  DF21 Overall EER: 11.9038% (bf=13357, sp=467178)
    D1: 6.1786%
    D2: 4.5364%
    D3: 13.3459%
    D4: 6.6317%
    D5: 14.7147%
    D6: 10.0539%
    D7: 15.6946%

--- lr=0.0001, r=16, alpha=16 ---
  ASV19 Overall EER: 5.4511% (bf=6623, sp=57489)
    D1: 4.5449%
    D2: 1.6965%
    D3: 2.3532%
    D4: 2.3354%
    D5: 7.7428%
    D6: 4.0258%
    D7: 13.2640%
  DF21 Overall EER: 11.1987% (bf=13357, sp=467178)
    D1: 6.2441%
    D2: 4.6746%
    D3: 13.4762%
    D4: 6.3802%
    D5: 14.3005%
    D6: 10.3237%
    D7: 17.2529%

--- lr=0.0001, r=4, alpha=8 ---
  [WARN] Missing: out/domain_lora/conformertcm/D7_lr0.0001_r4_a8/asv19_d7_scores.txt
  ASV19 Overall EER: 5.6735% (bf=5869, sp=51119)
    D1: 4.8430%
    D2: 1.6965%
    D3: 2.6173%
   

## Summary Table

In [5]:
all_results = aasist_results + conformer_results
df_results = pd.DataFrame(all_results)

# Pivot: model x HP x dataset → overall + domain EER
display_cols = ['model', 'lr', 'r', 'alpha', 'dataset', 'overall_eer'] + [f'D{d}' for d in range(1, 8)]
df_display = df_results[display_cols].round(4)
df_display

,model,lr,r,alpha,dataset,overall_eer,D1,D2,D3,D4,D5,D6,D7
0,aasist,5e-05,16,32,asv19,5.6919,3.9875,0.5202,2.7375,1.7666,8.9687,3.6069,10.2160
1,aasist,5e-05,16,32,df21,13.0875,6.0520,8.0573,15.2810,8.0667,14.8465,13.4957,16.8340
2,aasist,0.0001,32,32,asv19,4.8319,3.4890,0.6483,1.7205,1.3752,7.6044,3.3589,11.5385
3,aasist,0.0001,32,32,df21,12.1192,6.4400,5.2631,11.2612,8.0763,14.7782,9.9238,16.4987
4,aasist,5e-05,32,64,asv19,4.8618,4.4152,0.7842,1.5767,1.7666,8.2350,3.9945,9.6839
5,aasist,5e-05,32,64,df21,12.4891,6.4347,7.3422,9.9660,7.6524,15.9036,10.0587,15.4256
6,conformertcm,0.0001,16,32,asv19,5.7372,4.8352,1.4325,2.7375,2.1736,9.9178,3.6147,10.3372
7,conformertcm,0.0001,16,32,df21,11.9038,6.1786,4.5364,13.3459,6.6317,14.7147,10.0539,15.6946
8,conformertcm,0.0001,16,16,asv19,5.4511,4.5449,1.6965,2.3532,2.3354,7.7428,4.0258,13.2640
9,conformertcm,0.0001,16,16,df21,11.1987,6.2441,4.6746,13.4762,6.3802,14.3005,10.3237,17.2529


In [6]:
# CSV 저장
output_csv = 'results/domain_lora_overall_eer.csv'
os.makedirs('results', exist_ok=True)
df_display.to_csv(output_csv, index=False)
print(f'Saved to {output_csv}')

Saved to results/domain_lora_overall_eer.csv
